# Task 2: Test script

This notebook contains **domain data, expected results and checks**, not implementations of the reasoners. The checks become runnable when the `engine.py` provides the starter API.

**Scope:** Task 2 only:  Based on the supplied assignment, §§2.1.1, 2.1.2 and 2.2, and the starter's `applicable(wm, action, t)` / `step(wm, action, t)` interfaces. Backward planning (§2.3) is optional and excluded here. Task 1 is separate. This is only for testing, the final world and demonstration is in a separate notebook.

**Instructions:**

In each section: the cell is run before and after the implementation is ready to assert.

Rotate driver, predictor and reviewer. 

**Warning:** Unlike Task 1, do not keep firing all applicable actions until a fixpoint. Select **one action per time step**. Actions cause a new state; they do not merely infer more facts about the same state.

## 1. Representation summary

| Item | Contract used in this notebook |
|---|---|
| Stored fact | Ground tuple; integer time is the last argument, e.g. `("at", "obj1", "A", 0)` |
| Conditions | Timed patterns, all checked at the requested `t`; bind `?t` before matching |
| Action schema | Dictionary with `name`, `if`, `add`, `delete` |
| Effects | Grounded **untimed** tuples; engine stamps them with `t+1` |
| Applicable instance | Dictionary with grounded `name`, `add`, `delete`; optionally other fields |
| `step` input here | One grounded applicable instance, not an unbound schema |
| Deletion | Prevents a fact from being carried into the next state; history remains |
| Progress | Exactly one action and one advance from `t` to `t+1` per call |

Use a fresh branch of working memory when replaying a scenario. Do not append two alternative outcomes at the same time step into one history.

A held object's position is represented by `holding(obj1, t)` together with the hand's location. It has no separate `at(obj1, ..., t)` while held. This is a domain choice, not an engine special case.

In [14]:
# Knowledge fixture
from reasoning.domains import ACTIONS, TEST_2_FACTS, SCHEMAS

print("Initial facts:", len(TEST_2_FACTS), "| Action schemas:", len(ACTIONS))

Initial facts: 11 | Action schemas: 3


## 2. Establish expected test results

In [15]:
# Expected grounded instance, written by hand, not derived by a reasoner.
EXPECTED_GRAB = {
    "name": ("grab", "obj1", "A"),
    "add": [("holding", "obj1")],
    "delete": [("empty", "hand"), ("at", "obj1", "A")],
}

# Expected actions available at 0
EXPECTED_0_ACTIONS = {("grab", "obj1", "A"), ("move", "A", "B")}

## 3. Groundwork and connections

In [16]:
# Display/check helpers only: these are not the temporal engine.
def at_time(facts, t):
    return {fact for fact in facts if fact[-1] == t}

def contents(facts, t):
    """Untimed view for comparing two states without their timestamps."""
    return {fact[:-1] for fact in at_time(facts, t)}

def show_state(facts, t):
    state = at_time(facts, t)
    print(f"\nSTATE t={t} ({len(state)} facts)")
    for fact in sorted(state, key=repr):
        print(" ", fact)

def check_state(facts, t, expected):
    actual = at_time(facts, t)
    assert actual == expected, {
        "missing": expected - actual, "unexpected": actual - expected
    }
    print(f"PASS: exact state at t={t}")

show_state(TEST_2_FACTS, 0)


STATE t=0 (11 facts)
  ('at', 'hand', 'A', 0)
  ('at', 'obj1', 'A', 0)
  ('at', 'obj2', 'C', 0)
  ('empty', 'hand', 0)
  ('location', 'A', 0)
  ('location', 'B', 0)
  ('location', 'C', 0)
  ('object', 'obj1', 0)
  ('object', 'obj2', 0)
  ('route', 'A', 'B', 0)
  ('route', 'B', 'C', 0)


In [17]:
ENGINE_MODULE = "reasoning.engine"


import importlib
engine = importlib.import_module(ENGINE_MODULE)
WorkingMemory = engine.WorkingMemory
print("Engine loaded:", ENGINE_MODULE)



Engine loaded: reasoning.engine


In [18]:
# Helper functions for the tests



def apply_instance(wm, instance, t):
    # Contract: step mutates WM and returns that updated WM, as in the starter.
    result = engine.step(wm, instance, t)
    assert result is wm, "Adapt this wrapper if step returns a fresh WM."
    return result



def acceptance(label, test):
    test()
    print("PASS:", label)

## 3. Predict applicability at time 0

Expected action instances: **grab obj1 at A**, and **move A → B**. No release is available. `obj2` is at C, so it cannot be grabbed at A. Matching must carry the same `?loc` through all conditions.

Applicability should only read WM; it must not add effects, delete facts or advance time. Materialise the generator before changing WM to avoid iterating over a set while modifying it.

In [19]:

def test_initial_applicability():
    wm = WorkingMemory(TEST_2_FACTS)
    before = wm.snapshot()
    # Get the name of the available actions
    names = {tuple(a["name"]) for schema in ACTIONS for a in engine.instances(wm, schema, 0)}
    assert names == EXPECTED_0_ACTIONS, names
    grab = engine.choose(wm, ("grab", "obj1", "A"), SCHEMAS, 0)
    assert set(grab["add"]) == set(EXPECTED_GRAB["add"])
    assert set(grab["delete"]) == set(EXPECTED_GRAB["delete"])
    assert wm.snapshot() == before

acceptance("Grounded actions and read-only applicability", test_initial_applicability)

PASS: Grounded actions and read-only applicability


## 4. Test 2.1.1

**Predict:** if we only assert the grab's add set at time 1, what remains at time 1?

Expected: **only `holding(obj1, 1)`**. The hand's position, obj2's location, object declarations, locations and routes are absent at time 1. The time-0 facts may still be in the history, but they do not establish anything at time 1.

The following intentionally incomplete baseline is demonstration code. Do not mistake it for the correct `step` implementation. It uses the hand-written grab instance so you can record the frame problem before your engine is ready.

In [20]:
effects_only_history = TEST_2_FACTS | {
    effect + (1,) for effect in EXPECTED_GRAB["add"]
}
show_state(effects_only_history, 1)
assert at_time(effects_only_history, 1) == {("holding", "obj1", 1)}
assert ("at", "obj2", "C", 0) in effects_only_history
assert ("at", "obj2", "C", 1) not in effects_only_history
print("EXPECTED FAILURE RECORDED: untouched world facts are missing at t=1.")

def test_effects_only_stalls():
    wm = WorkingMemory(effects_only_history)
    assert not engine.instances(wm, SCHEMAS["move"], 1)
    assert not engine.instances(wm, SCHEMAS["release"], 1)

acceptance("Effects-only world cannot move or release at t=1", test_effects_only_stalls)


STATE t=1 (1 facts)
  ('holding', 'obj1', 1)
EXPECTED FAILURE RECORDED: untouched world facts are missing at t=1.
PASS: Effects-only world cannot move or release at t=1


## 5. Persistence: predict the complete next state (§2.1.2)

Strip the current facts' time arguments to form `Sₜ`. For one grounded action:

**Sₜ₊₁ = (Sₜ − delete) ∪ add**, with all resulting facts stamped `t+1`.

This describes the required semantics; your team implements the generic engine loop. Old stamped facts stay in WM as history. Deleting `empty(hand)` means it is absent at time 1, not that `empty(hand, 0)` is erased.

**Predict:** after grabbing obj1, there are 10 facts at time 1. The hand is still at A, holding obj1; obj1 has no separate location, and obj2 remains at C. All declarations and routes survive.

In [21]:
# Hand-written expected worlds: independent of the engine under test.
BACKGROUND = {
    ("object", "obj1"), ("object", "obj2"),
    ("location", "A"), ("location", "B"), ("location", "C"),
    ("route", "A", "B"), ("route", "B", "C"),
}
EXPECTED_CONTENTS = {
    0: BACKGROUND | {("at", "hand", "A"), ("at", "obj1", "A"),
                     ("at", "obj2", "C"), ("empty", "hand")},
    1: BACKGROUND | {("at", "hand", "A"), ("holding", "obj1"),
                     ("at", "obj2", "C")},
    2: BACKGROUND | {("at", "hand", "B"), ("holding", "obj1"),
                     ("at", "obj2", "C")},
    3: BACKGROUND | {("at", "hand", "B"), ("at", "obj1", "B"),
                     ("at", "obj2", "C"), ("empty", "hand")},
}
EXPECTED_STATES = {
    t: {fact + (t,) for fact in state} for t, state in EXPECTED_CONTENTS.items()
}
assert EXPECTED_STATES[0] == TEST_2_FACTS
for t in range(4):
    print(f"Expected t={t}: {len(EXPECTED_STATES[t])} facts")

def test_one_step():
    wm = WorkingMemory(TEST_2_FACTS)
    apply_instance(wm, engine.choose(wm, ("grab", "obj1", "A"),SCHEMAS, 0),  0)
    show_state(wm.facts, 1)
    check_state(wm.facts, 1, EXPECTED_STATES[1])
    assert wm.snapshot() == EXPECTED_STATES[0] | EXPECTED_STATES[1]

acceptance("Grab with generic persistence and preserved history", test_one_step)

Expected t=0: 11 facts
Expected t=1: 10 facts
Expected t=2: 10 facts
Expected t=3: 11 facts

STATE t=1 (10 facts)
  ('at', 'hand', 'A', 1)
  ('at', 'obj2', 'C', 1)
  ('holding', 'obj1', 1)
  ('location', 'A', 1)
  ('location', 'B', 1)
  ('location', 'C', 1)
  ('object', 'obj1', 1)
  ('object', 'obj2', 1)
  ('route', 'A', 'B', 1)
  ('route', 'B', 'C', 1)
PASS: exact state at t=1
PASS: Grab with generic persistence and preserved history


## 6. Multi-step demonstration (§2.2)

Predict each row before running. This is a chosen sequence, not an automatically discovered plan.

| State | Action just completed | Hand | obj1 | obj2 | Hand status |
|---|---|---|---|---|---|
| 0 | Initial | A | A | C | Empty |
| 1 | Grab obj1 at A | A | Held | C | Holding obj1 |
| 2 | Move A → B | B | Held | C | Holding obj1 |
| 3 | Release obj1 at B | B | B | C | Empty |

**Target:** `at(obj1, B, 3)` while `at(obj2, C, 3)` still holds. Print the complete state after every action, as the assignment requests.

In [22]:
SEQUENCE = [("grab", "obj1", "A"), ("move", "A", "B"),
            ("release", "obj1", "B")]

def test_sequence():
    wm = WorkingMemory(TEST_2_FACTS)
    show_state(wm.facts, 0)
    for t, action_name in enumerate(SEQUENCE):
        old_history = wm.snapshot()
        action = engine.choose(wm, action_name, SCHEMAS, t)
        print("\nFIRE:", action["name"], f"at t={t}")
        print("  Added at next step:", action["add"])
        print("  Not carried forward:", action["delete"])
        apply_instance(wm, action, t)
        show_state(wm.facts, t + 1)
        check_state(wm.facts, t + 1, EXPECTED_STATES[t + 1])
        assert old_history <= wm.facts, "History was erased"
        assert wm.snapshot() == set().union(*(EXPECTED_STATES[i] for i in range(t + 2)))
        assert ("at", "obj2", "C", t + 1) in wm.facts
    assert wm.holds(("at", "obj1", "B", 3))
    assert wm.holds(("at", "obj2", "C", 3))
    assert not wm.holds(("holding", "obj1", 3))
    print("TARGET REACHED; untouched object preserved at every step.")
    return wm

acceptance("Grab → move → release; exact states and target", test_sequence)


STATE t=0 (11 facts)
  ('at', 'hand', 'A', 0)
  ('at', 'obj1', 'A', 0)
  ('at', 'obj2', 'C', 0)
  ('empty', 'hand', 0)
  ('location', 'A', 0)
  ('location', 'B', 0)
  ('location', 'C', 0)
  ('object', 'obj1', 0)
  ('object', 'obj2', 0)
  ('route', 'A', 'B', 0)
  ('route', 'B', 'C', 0)

FIRE: ('grab', 'obj1', 'A') at t=0
  Added at next step: [('holding', 'obj1')]
  Not carried forward: [('empty', 'hand'), ('at', 'obj1', 'A')]

STATE t=1 (10 facts)
  ('at', 'hand', 'A', 1)
  ('at', 'obj2', 'C', 1)
  ('holding', 'obj1', 1)
  ('location', 'A', 1)
  ('location', 'B', 1)
  ('location', 'C', 1)
  ('object', 'obj1', 1)
  ('object', 'obj2', 1)
  ('route', 'A', 'B', 1)
  ('route', 'B', 'C', 1)
PASS: exact state at t=1

FIRE: ('move', 'A', 'B') at t=1
  Added at next step: [('at', 'hand', 'B')]
  Not carried forward: [('at', 'hand', 'A')]

STATE t=2 (10 facts)
  ('at', 'hand', 'B', 2)
  ('at', 'obj2', 'C', 2)
  ('holding', 'obj1', 2)
  ('location', 'A', 2)
  ('location', 'B', 2)
  ('location', 

## 7. Check time isolation and unavailable actions

History must not make a past condition count as a present condition. At time 2, `empty(hand, 0)` exists, but `empty(hand, 2)` does not. The hand cannot grab another object while holding obj1. After release at time 3, it cannot release obj1 again.

The minimum contract here is that invalid instances are **not yielded by `applicable`**. Whether `step` also rejects an invalid or stale instance is a team API choice; if you promise rejection, add a test that it leaves WM unchanged on failure.

In [23]:
def test_time_isolation():
    wm = WorkingMemory(TEST_2_FACTS)
    apply_instance(wm, engine.choose(wm, SEQUENCE[0], SCHEMAS,0), 0)
    assert not engine.instances(wm, SCHEMAS["grab"], 1)
    apply_instance(wm, engine.choose(wm, SEQUENCE[1], SCHEMAS, 1), 1)
    assert not engine.instances(wm, SCHEMAS["grab"], 2)
    assert {tuple(a["name"]) for a in engine.instances(wm, SCHEMAS["move"], 2)} == {
        ("move", "B", "C")
    }, "Historical A must not enable another A → B move"
    apply_instance(wm, engine.choose(wm, SEQUENCE[2],SCHEMAS, 2), 2)
    assert not engine.instances(wm, SCHEMAS["release"], 3)
    assert list(wm.ask(("at", "obj1", "?where", 0))) == [{"?where": "A"}]
    assert list(wm.ask(("at", "obj1", "?where", 3))) == [{"?where": "B"}]
    assert list(wm.ask(("at", "obj2", "?where", 1))) == [{"?where": "C"}]

acceptance("Historical facts do not satisfy current preconditions", test_time_isolation)

PASS: Historical facts do not satisfy current preconditions


## 8. Variables and multiple objects. same schemas, different world

Resetting to a fresh scenario where the hand and **both objects** start at C. Expecting two grab instances: one for obj1 and one for obj2. Select obj2. The next state must contain `holding(obj2, 1)` and preserve `at(obj1, C, 1)`.

This catches an implementation that always chooses obj1 or stops after the first condition match. No engine changes are allowed for this scenario.

In [24]:
def test_multiple_objects():
    facts = (TEST_2_FACTS - {("at", "hand", "A", 0), ("at", "obj1", "A", 0)}) | {
        ("at", "hand", "C", 0), ("at", "obj1", "C", 0)
    }
    wm = WorkingMemory(facts)
    options = engine.instances(wm, SCHEMAS["grab"], 0)
    assert {tuple(a["name"]) for a in options} == {
        ("grab", "obj1", "C"), ("grab", "obj2", "C")
    }
    selected = next(a for a in options if tuple(a["name"]) == ("grab", "obj2", "C"))
    apply_instance(wm, selected, 0)
    assert wm.holds(("holding", "obj2", 1))
    assert wm.holds(("at", "obj1", "C", 1))
    assert not wm.holds(("at", "obj2", "C", 1))
    assert not wm.holds(("empty", "hand", 1))

acceptance("All bindings and either object supported", test_multiple_objects)

PASS: All bindings and either object supported


## 9. Engine/knowledge separation with unfamiliar facts and a new action

A persistence engine should carry any timed fact, not just known predicates. Add `inspection_tag(obj2, blue, 0)`: it must survive the grab. Then add a **new action dictionary** that changes that tag, and confirm that the generic engine handles it without any code change.

This small extension tests both generic persistence and generic deletion. It is an acceptance check, not an extra assignment feature you need to present.

In [25]:
RETAG_ACTION = {
    "name": ("retag", "?o"),
    "if": [("object", "?o", "?t"), ("inspection_tag", "?o", "blue", "?t")],
    "add": [("inspection_tag", "?o", "green")],
    "delete": [("inspection_tag", "?o", "blue")],
}

def test_generic_data():
    wm = WorkingMemory(TEST_2_FACTS | {("inspection_tag", "obj2", "blue", 0)})
    apply_instance(wm, engine.choose(wm, ("grab", "obj1", "A"), SCHEMAS, 0), 0)
    assert wm.holds(("inspection_tag", "obj2", "blue", 1))
    options = engine.instances(wm, RETAG_ACTION, 1)
    assert len(options) == 1
    assert tuple(options[0]["name"]) == ("retag", "obj2")
    apply_instance(wm, options[0], 1)
    expected = (EXPECTED_CONTENTS[1] | {("inspection_tag", "obj2", "green")})
    assert contents(wm.facts, 2) == expected
    assert wm.holds(("inspection_tag", "obj2", "blue", 1))
    assert not wm.holds(("inspection_tag", "obj2", "blue", 2))

acceptance("Unfamiliar predicate and new action handled as data", test_generic_data)

PASS: Unfamiliar predicate and new action handled as data
